mlflow-energyforecast

This is a showcase for ML Flow capabilities, based on the article http://the-odd-dataguy.com/be-more-efficient-to-produce-ml-models-with-mlflow and a github https://github.com/jeanmidevacc/mlflow-energyforecast

# Please check the requirements.in file for more details
!pip install -r requirements.txt

!pip install pandas --upgrade --user
!pip install mlflow --upgrade --user
!pip install joblib --upgrade --user
!pip install numpy --upgrade --user 
!pip install scipy --upgrade --user 
!pip install scikit-learn --upgrade --user
!pip install boto3 --upgrade --user

In [10]:
import os
import warnings
import time
import json
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from minio import Minio
from minio.error import BucketAlreadyOwnedByYou
from mlflow.models.signature import infer_signature
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score
from sklearn.exceptions import ConvergenceWarning
from tenacity import retry, stop_after_attempt, wait_exponential
from joblib import Parallel, delayed


# suppress warnings
warnings.filterwarnings("ignore")

In [13]:
MINIO_HOST = os.environ["MINIO_ENDPOINT_URL"].split("http://")[1]
MINIO_BUCKET = "mlflow-test-mldm"

In [15]:
# Initialize a MinIO client
mc = Minio(
    endpoint=MINIO_HOST,
    access_key=os.environ["AWS_ACCESS_KEY_ID"],
    secret_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    secure=False,
)

try:
    mc.make_bucket(MINIO_BUCKET)
except BucketAlreadyOwnedByYou:
    print(f"Bucket {MINIO_BUCKET} already exists!")

Bucket mlflow-test-mldm already exists!


Data preparation

In [16]:
# Collect the data 
df_nationalconsumption_electricity_daily = pd.read_csv("https://raw.githubusercontent.com/jeanmidevacc/mlflow-energyforecast/master/data/rtu_data.csv")
df_nationalconsumption_electricity_daily.set_index(["day"], inplace = True)

In [17]:
# Prepare the training set and the testing set
df_trainvalidate_energyconsumption = df_nationalconsumption_electricity_daily[df_nationalconsumption_electricity_daily["datastatus"] == "Définitif"]
del df_trainvalidate_energyconsumption["datastatus"]

df_test_energyconsumption = df_nationalconsumption_electricity_daily[df_nationalconsumption_electricity_daily["datastatus"] == "Consolidé"]
del df_test_energyconsumption["datastatus"]

print("Size of the training set : ",len(df_trainvalidate_energyconsumption))
print("Size of the testing set : ",len(df_test_energyconsumption))


Size of the training set :  1081
Size of the testing set :  233


In [18]:
# Define the inputs and the output
output = "dailyconsumption"
allinputs = list(df_trainvalidate_energyconsumption.columns)
allinputs.remove(output)

print("Output to predict : ", output)
print("Inputs for the prediction : ", allinputs)

Output to predict :  dailyconsumption
Inputs for the prediction :  ['weekday', 'week', 'month', 'year', 'avg_min_temperature', 'avg_max_temperature', 'avg_mean_temperature', 'wavg_min_temperature', 'wavg_max_temperature', 'wavg_mean_temperature', 'is_holiday']


In [19]:
# Build different set of featurws for the model
possible_inputs = {
    "all" : allinputs,
    "only_allday_inputs" : ["weekday", "month", "is_holiday", "week"],
    "only_allweatheravg_inputs" : ["avg_min_temperature", "avg_max_temperature", "avg_mean_temperature","wavg_min_temperature", "wavg_max_temperature", "wavg_mean_temperature"],
    "only_meanweather_inputs_avg" : ["avg_mean_temperature"],
    "only_meanweather_inputs_wavg" : ["wavg_mean_temperature"],
}

In [20]:
# Prepare the output of the model
array_output_train = np.array(df_trainvalidate_energyconsumption[output])
array_output_test = np.array(df_test_energyconsumption[output])

In [32]:
experiment_name = "electricityconsumption-forecast"
experiment = mlflow.get_experiment_by_name(experiment_name)
experiment_id = (
    mlflow.create_experiment(name=experiment_name)
    if experiment is None
    else experiment.experiment_id
)

# check that the experiment was created successfully
assert (
    mlflow.get_experiment(experiment_id).name == experiment_name
), f"Failed to create experiment {experiment_name}!"

In [33]:
# Define the evaluation function that will do the computation of the different metrics of accuracy (RMSE,MAE,R2)
def evaluation_model(y_test, y_pred):

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    metrics = {
        "rmse" : rmse,
        "r2" : r2,
        "mae" : mae,
    }
    
    return metrics

KNN regressor

In [36]:
from sklearn.neighbors import KNeighborsRegressor

def train_knnmodel(parameters, inputs, tags, log = False):
    with mlflow.start_run(experiment_id=experiment_id, nested=True):
        
        # Prepare the data
        array_inputs_train = np.array(df_trainvalidate_energyconsumption[inputs])
        array_inputs_test = np.array(df_test_energyconsumption[inputs])
        
        
        # Build the model
        tic = time.time()
        model = KNeighborsRegressor(parameters["nbr_neighbors"], weights = parameters["weight_method"])
        model.fit(array_inputs_train, array_output_train)
        duration_training = time.time() - tic

        # Make the prediction
        tic1 = time.time()
        prediction = model.predict(array_inputs_test)
        duration_prediction = time.time() - tic1

        # Evaluate the model prediction
        metrics = evaluation_model(array_output_test, prediction)

        # Log in the console
        if log:
            print(f"KNN regressor:")
            print(parameters)
            print(metrics)

        # Log in mlflow (parameter)
        mlflow.log_params(parameters)

        # Log in mlflow (metrics)
        metrics["duration_training"] = duration_training
        metrics["duration_prediction"] = duration_prediction
        mlflow.log_metrics(metrics)

        # log in mlflow (model)
        mlflow.sklearn.log_model(model, f"model")
                
        # Tag the model
        mlflow.set_tags(tags)



In [37]:
# Test the different combinations
configurations = []
for nbr_neighbors in [1,2,5,10]:
    for weight_method in ['uniform','distance']:
        for field in possible_inputs:
            parameters = {
                "nbr_neighbors" : nbr_neighbors,
                "weight_method" : weight_method
            }

            tags = {
                "model" : "knn",
                "inputs" : field
            }
            
            configurations.append([parameters, tags])

            train_knnmodel(parameters, possible_inputs[field], tags)


2025/02/17 21:08:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run learned-sponge-895 at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5/runs/8287a6f06d174c17988b833a9e5254d5
🧪 View experiment at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5


2025/02/17 21:08:54 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run debonair-kit-228 at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5/runs/c6d2e7e290c8432e95c14ccc086caed9
🧪 View experiment at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5


2025/02/17 21:08:56 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


...

🏃 View run beautiful-crow-759 at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5/runs/74dd497b95e6469fb980438191e872b4
🧪 View experiment at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5


2025/02/17 21:09:48 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run grandiose-deer-702 at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5/runs/519c3dae972e4db68383e6656a7f9c68
🧪 View experiment at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5


2025/02/17 21:09:49 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run indecisive-cat-83 at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5/runs/9424f17bf77f4c61a7c9ffbaae448c65
🧪 View experiment at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5


MLP regressor

In [39]:
from sklearn.neural_network import MLPRegressor

def train_mlpmodel(parameters, inputs, tags, log = False):
    with mlflow.start_run(experiment_id=experiment_id, nested=True):
        
        # Prepare the data
        array_inputs_train = np.array(df_trainvalidate_energyconsumption[inputs])
        array_inputs_test = np.array(df_test_energyconsumption[inputs])
        
        # Build the model
        tic = time.time()

        model = MLPRegressor(
            hidden_layer_sizes = parameters["hidden_layers"],
            activation = parameters["activation"],
            solver = parameters["solver"],
            max_iter = parameters["nbr_iteration"],
            random_state = 0)
        
        model.fit(array_inputs_train, array_output_train)
        duration_training = time.time() - tic

        # Make the prediction
        tic1 = time.time()
        prediction = model.predict(array_inputs_test)
        duration_prediction = time.time() - tic1

        # Evaluate the model prediction
        metrics = evaluation_model(array_output_test, prediction)

        # Log in the console
        if log:
            print(f"Random forest regressor:")
            print(parameters)
            print(metrics)
    
        # Log in mlflow (parameter)
        mlflow.log_params(parameters)

        # Log in mlflow (metrics)
        metrics["duration_training"] = duration_training
        metrics["duration_prediction"] = duration_prediction
        mlflow.log_metrics(metrics)

        # log in mlflow (model)
        mlflow.sklearn.log_model(model, f"model")
        
        # Tag the model
        mlflow.set_tags(tags)   

In [40]:
for hiddenlayers in [4,8,16]:
    for activation in ["identity","logistic",]:
        for solver in ["lbfgs"]:
            for nbriteration in [10,100,1000]:
                for field in possible_inputs:
                    parameters = {
                        "hidden_layers" : hiddenlayers,
                        "activation" : activation,
                        "solver" : solver,
                        "nbr_iteration" : nbriteration
                    }

                    tags = {
                        "model" : "mlp",
                        "inputs" : field
                    }

                    train_mlpmodel(parameters, possible_inputs[field], tags)


2025/02/17 21:11:49 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run mysterious-ray-75 at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5/runs/27bc22add086482f9b131ab5155178b0
🧪 View experiment at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5


2025/02/17 21:11:50 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run calm-worm-109 at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5/runs/3c8bcf12096a440b9f3cf64947ab094b
🧪 View experiment at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5


2025/02/17 21:11:52 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


...

🏃 View run inquisitive-fish-49 at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5/runs/f7467c6acc864f3d841353c0a010b129
🧪 View experiment at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5


2025/02/17 21:13:57 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run lyrical-duck-119 at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5/runs/8d38c2d8567a42a4bf1144203ffcc10c
🧪 View experiment at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5


2025/02/17 21:13:59 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run fortunate-vole-978 at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5/runs/389bbd9f77d94038ac873a0364629a52
🧪 View experiment at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5


Use a handmade model (scipy approach)

In [42]:
class PTG:
    def __init__(self, thresholds_x0, thresholds_a, thresholds_b):
        self.thresholds_x0 = thresholds_x0
        self.thresholds_a = thresholds_a
        self.thresholds_b = thresholds_b
        
    def get_ptgmodel(self, x, a, b, x0):
        return np.piecewise(x, [x < x0, x >= x0], [lambda x: a*x + b , lambda x : a*x0 + b])
        
    def fit(self, dfx, y):
        x = np.array(dfx)
        
        # Define the bounds
        bounds_min = [thresholds_a[0], thresholds_b[0], thresholds_x0[0]]
        bounds_max = [thresholds_a[1], thresholds_b[1], thresholds_x0[1]]
        bounds = (bounds_min, bounds_max)

        # Fit a model
        popt, pcov = scipy.optimize.curve_fit(self.get_ptgmodel, x, y, bounds = bounds)

        # Get the parameter of the model
        a = popt[0]
        b = popt[1]
        x0 = popt[2]
        
        self.coefficients = [a, b, x0]
        
    def predict(self,dfx):
        x = np.array(dfx)
        predictions = []
        for elt in x:
            forecast = self.get_ptgmodel(elt, self.coefficients[0], self.coefficients[1], self.coefficients[2])
            predictions.append(forecast)
        return np.array(predictions)
        
def train_ptgmodel(parameters, inputs, tags, log = False):
    with mlflow.start_run(experiment_id=experiment_id, nested=True):
        
        # Prepare the data
        df_inputs_train = df_trainvalidate_energyconsumption[inputs[0]]
        df_inputs_test = df_test_energyconsumption[inputs[0]]
        
        
        # Build the model
        tic = time.time()
        
        model = PTG(parameters["thresholds_x0"], parameters["thresholds_a"], parameters["thresholds_b"])
        
        model.fit(df_inputs_train, array_output_train)
        duration_training = time.time() - tic

        # Make the prediction
        tic1 = time.time()
        prediction = model.predict(df_inputs_test)
        duration_prediction = time.time() - tic1

        # Evaluate the model prediction
        metrics = evaluation_model(array_output_test, prediction)

        # Log in the console
        if log:
            print(f"PTG:")
            print(parameters)
            print(metrics)
    
        # Log in mlflow (parameter)
        mlflow.log_params(parameters)  

        # Log in mlflow (metrics)
        metrics["duration_training"] = duration_training
        metrics["duration_prediction"] = duration_prediction
        mlflow.log_metrics(metrics)

        # log in mlflow (model)
        mlflow.sklearn.log_model(model, f"model")
        
        # Tag the model
        mlflow.set_tags(tags)           


In [43]:
# Define the parameters of the model
thresholds_x0 = [0, 20]
thresholds_a = [-200000, -50000]
thresholds_b = [1000000, 3000000]

parameters = {
    "thresholds_x0" : thresholds_x0,
    "thresholds_a" : thresholds_a,
    "thresholds_b" : thresholds_b
}

for field in ["only_meanweather_inputs_avg", "only_meanweather_inputs_wavg"]:
    
    tags = {
        "model" : "ptg",
        "inputs" : field
    }
    
    train_ptgmodel(parameters, possible_inputs[field], tags, log = False)

2025/02/17 21:17:12 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run delightful-pug-831 at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5/runs/323cda84ca924fdabd97a22aee974d28
🧪 View experiment at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5


2025/02/17 21:17:13 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run capricious-eel-563 at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5/runs/413b15f8c38448a7b4bae51cf7a10682
🧪 View experiment at: http://mlflow-server.kubeflow.svc.cluster.local:5000/#/experiments/5


Evaluate mlflow results

In [45]:
# Select the run of the experiment
df_runs = mlflow.search_runs(experiment_ids=experiment_id)
print("Number of runs done : ", len(df_runs))

Number of runs done :  132


In [46]:
# Quick sorting to get the best models based on the RMSE metric
df_runs.sort_values(["metrics.rmse"], ascending = True, inplace = True)
df_runs.head()

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.r2,metrics.rmse,metrics.duration_prediction,metrics.mae,...,params.nbr_iteration,params.nbr_neighbors,params.weight_method,tags.inputs,tags.model,tags.mlflow.source.type,tags.mlflow.source.name,tags.mlflow.runName,tags.mlflow.log-model.history,tags.mlflow.user
106,9803229d47974dfc802e6ca91060bde9,5,FINISHED,s3://mlflow/5/9803229d47974dfc802e6ca91060bde9...,2025-02-17 21:09:28.193000+00:00,2025-02-17 21:09:29.604000+00:00,0.935956,134649.399348,0.001047,104040.339809,...,None,5,distance,all,knn,LOCAL,/opt/conda/lib/python3.11/site-packages/ipyker...,orderly-hawk-862,"[{""run_id"": ""9803229d47974dfc802e6ca91060bde9""...",jovyan
96,f694defc9b1847e690281ce3b3c886bf,5,FINISHED,s3://mlflow/5/f694defc9b1847e690281ce3b3c886bf...,2025-02-17 21:09:42.289000+00:00,2025-02-17 21:09:43.719000+00:00,0.935111,135534.759873,0.001222,105833.358681,...,None,10,distance,all,knn,LOCAL,/opt/conda/lib/python3.11/site-packages/ipyker...,puzzled-ant-765,"[{""run_id"": ""f694defc9b1847e690281ce3b3c886bf""...",jovyan
111,9b9e21b2fd7441e4838042cdc867d680,5,FINISHED,s3://mlflow/5/9b9e21b2fd7441e4838042cdc867d680...,2025-02-17 21:09:21.203000+00:00,2025-02-17 21:09:22.558000+00:00,0.934465,136207.422483,0.001046,105793.727897,...,None,5,uniform,all,knn,LOCAL,/opt/conda/lib/python3.11/site-packages/ipyker...,vaunted-sloth-983,"[{""run_id"": ""9b9e21b2fd7441e4838042cdc867d680""...",jovyan
101,3d0d6744e9e54dbb85fe12261b748798,5,FINISHED,s3://mlflow/5/3d0d6744e9e54dbb85fe12261b748798...,2025-02-17 21:09:35.213000+00:00,2025-02-17 21:09:36.592000+00:00,0.932457,138279.158616,0.001230,108427.970386,...,None,10,uniform,all,knn,LOCAL,/opt/conda/lib/python3.11/site-packages/ipyker...,placid-skink-484,"[{""run_id"": ""3d0d6744e9e54dbb85fe12261b748798""...",jovyan
116,e0ca1ffb856c4ed39b29d2bb06f8fd80,5,FINISHED,s3://mlflow/5/e0ca1ffb856c4ed39b29d2bb06f8fd80...,2025-02-17 21:09:14.261000+00:00,2025-02-17 21:09:15.579000+00:00,0.921697,148886.582823,0.000929,114048.572635,...,None,2,distance,all,knn,LOCAL,/opt/conda/lib/python3.11/site-packages/ipyker...,abrasive-carp-798,"[{""run_id"": ""e0ca1ffb856c4ed39b29d2bb06f8fd80""...",jovyan


In [47]:
# Get the best one
runid_selected = df_runs.head(1)["run_id"].values[0]
runid_selected

'9803229d47974dfc802e6ca91060bde9'